# Debug: Matching Verification

Run the cells below to verify the matching is correct

In [1]:
import sys
sys.path.insert(0, '/Users/tduani/PycharmProjects/distributed_node_matching')

import random
from src.graph import GraphManager
from src.algorithms.implementations import ItaiIsraeliMaximalMatching
from src.simulation import Scheduler, SimulationConfig

# Same parameters as visualization notebook
NODE_COUNT = 8
EDGE_DENSITY = 0.4
GRAPH_TYPE = 'random'
SEED = 42
MAX_ROUNDS = 1000

random.seed(SEED)

# Generate graph
graph = GraphManager.create_empty_graph()
for i in range(1, NODE_COUNT + 1):
    graph.add_vertex(i)

vertex_list = list(range(1, NODE_COUNT + 1))
edges_added = set()

max_possible_edges = NODE_COUNT * (NODE_COUNT - 1) // 2
target_edges = int(max_possible_edges * EDGE_DENSITY)

while len(edges_added) < target_edges:
    u = random.choice(vertex_list)
    v = random.choice(vertex_list)
    if u != v and (min(u, v), max(u, v)) not in edges_added:
        weight = random.uniform(0.0, 1.0)
        graph.add_edge(u, v, weight)
        edges_added.add((min(u, v), max(u, v)))

# Run algorithm
algo = ItaiIsraeliMaximalMatching(seed=SEED)
config = SimulationConfig(max_rounds=MAX_ROUNDS, collect_snapshots=True, random_seed=SEED)
scheduler = Scheduler(graph, algo, config)
rounds = scheduler.run_until_termination()

matching = scheduler.final_matching

print("✅ Graph and algorithm initialized")

✅ Graph and algorithm initialized


In [2]:
print("\n" + "="*70)
print("DEBUG: Graph Structure and Matching Validation")
print("="*70)

# Print all edges
print("\n📊 ALL EDGES IN GRAPH:")
all_edges = sorted(list(graph._graph.edges()))
for u, v in all_edges:
    print(f"  {u} -- {v}")

print(f"\nTotal edges: {len(all_edges)}")

# Print matching
print(f"\n🔗 FINAL MATCHING DICTIONARY:")
print(f"  {matching}")
print(f"  Total entries: {len(matching)}")
print(f"  Pairs: {len(matching) // 2}")


DEBUG: Graph Structure and Matching Validation

📊 ALL EDGES IN GRAPH:
  1 -- 2
  1 -- 4
  1 -- 5
  2 -- 3
  2 -- 4
  2 -- 6
  2 -- 7
  3 -- 4
  3 -- 7
  4 -- 7
  5 -- 6

Total edges: 11

🔗 FINAL MATCHING DICTIONARY:
  {1: 5, 2: 7, 3: 4, 4: 3, 5: 1, 7: 2}
  Total entries: 6
  Pairs: 3


In [3]:
# Validate matching
print("\n✓ MATCHING VALIDATION:")
invalid = False
for u, v in matching.items():
    if matching.get(v) != u:
        print(f"  ❌ ERROR: {u}→{v} but {v}→{matching.get(v)}")
        invalid = True

if not invalid:
    print(f"  ✅ All {len(matching)//2} pairs are bidirectional")
else:
    print(f"  ❌ MATCHING IS INVALID!")


✓ MATCHING VALIDATION:
  ✅ All 3 pairs are bidirectional


In [4]:
# Which edges will be colored green (matched)
print(f"\n🟢 EDGES THAT WILL BE GREEN (matched):")
green_edges = []
for u, v in all_edges:
    if (matching.get(u) == v and matching.get(v) == u):
        green_edges.append((u, v))
        print(f"  {u}-{v}: matching[{u}]={matching.get(u)}, matching[{v}]={matching.get(v)} ✓")

if not green_edges:
    print("  (none)")

print(f"\n  Total green edges: {len(green_edges)}")


🟢 EDGES THAT WILL BE GREEN (matched):
  1-5: matching[1]=5, matching[5]=1 ✓
  2-7: matching[2]=7, matching[7]=2 ✓
  3-4: matching[3]=4, matching[4]=3 ✓

  Total green edges: 3


In [5]:
# Which edges will be colored gray (not matched)
print(f"\n⚫ EDGES THAT WILL BE GRAY (not matched):")
gray_edges = []
for u, v in all_edges:
    if not (matching.get(u) == v and matching.get(v) == u):
        gray_edges.append((u, v))
        match_u = matching.get(u)
        match_v = matching.get(v)
        print(f"  {u}-{v}: matching[{u}]={match_u}, matching[{v}]={match_v}")

print(f"\n  Total gray edges: {len(gray_edges)}")


⚫ EDGES THAT WILL BE GRAY (not matched):
  1-2: matching[1]=5, matching[2]=7
  1-4: matching[1]=5, matching[4]=3
  2-3: matching[2]=7, matching[3]=4
  2-4: matching[2]=7, matching[4]=3
  2-6: matching[2]=7, matching[6]=None
  3-7: matching[3]=4, matching[7]=2
  4-7: matching[4]=3, matching[7]=2
  5-6: matching[5]=1, matching[6]=None

  Total gray edges: 8


In [6]:
# Check nodes
matched_nodes = set(matching.keys())
all_nodes = set(range(1, NODE_COUNT + 1))
unmatched_nodes = all_nodes - matched_nodes

print(f"\n👥 NODES:")
print(f"  Matched nodes ({len(matched_nodes)}): {sorted(matched_nodes)}")
print(f"  Unmatched nodes ({len(unmatched_nodes)}): {sorted(unmatched_nodes)}")

# Can unmatched nodes be matched?
print(f"\n🔍 CAN UNMATCHED NODES BE MATCHED?")
can_match = False
for node in sorted(unmatched_nodes):
    neighbors = list(graph.neighbors(node))
    unmatched_neighbors = [n for n in neighbors if n in unmatched_nodes]
    print(f"  Node {node}: neighbors={neighbors}, unmatched_neighbors={unmatched_neighbors}")
    if unmatched_neighbors:
        can_match = True

if can_match:
    print(f"\n⚠️  WARNING: More matching is possible! Algorithm may not be optimal.")
else:
    print(f"\n✅ Matching is MAXIMAL - no more pairs can be added.")


👥 NODES:
  Matched nodes (6): [1, 2, 3, 4, 5, 7]
  Unmatched nodes (2): [6, 8]

🔍 CAN UNMATCHED NODES BE MATCHED?
  Node 6: neighbors=[2, 5], unmatched_neighbors=[]
  Node 8: neighbors=[], unmatched_neighbors=[]

✅ Matching is MAXIMAL - no more pairs can be added.


In [7]:
# Summary
print(f"\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Nodes: {NODE_COUNT}")
print(f"Edges: {len(all_edges)}")
print(f"Matched pairs: {len(matching) // 2}")
print(f"Unmatched nodes: {len(unmatched_nodes)}")
print(f"Convergence: {rounds} rounds")
print(f"\nMatching is {'✅ VALID' if not invalid else '❌ INVALID'}")
print(f"Matching is {'✅ MAXIMAL' if not can_match else '❌ NOT MAXIMAL'}")


SUMMARY
Nodes: 8
Edges: 11
Matched pairs: 3
Unmatched nodes: 2
Convergence: 101 rounds

Matching is ✅ VALID
Matching is ✅ MAXIMAL
